<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/4_4_Bayesian_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part IV. Generalized Linear Models, Bayesian Methods, and Latent Variables


## Naïve Bayes Classifier

## Generalized Linear Models (GLM)

## Bayesian Linear Regression

## Bayesian Logistic Regression

## Latent Variable Models: EM Algorithm and GMM

### Вывод MAP-оценки методом Ньютона–Рафсона и апостериорная ковариация

После того как мы установили необходимость приближённых методов и описали лапласовское приближение, перейдём к практическому вычислению его параметров — моды апостериорного распределения (MAP) и соответствующей ковариационной матрицы. Центральную роль здесь играет метод Ньютона–Рафсона, который для логистической регрессии эквивалентен итеративно перевзвешенному методу наименьших квадратов (IRLS). Этот подход одновременно даёт и точечную оценку, и гессиан, необходимый для аппроксимации апостериорной неопределённости.

#### 1. Логарифм апостериорного распределения и его градиент

Исходим из апостериорной плотности с гауссовским априорным $\mathbf{w} \sim \mathcal{N}(\mathbf{0}, \sigma_w^2 \mathbf{I})$ и бернуллиевским правдоподобием. Логарифм апостериорной плотности с точностью до аддитивной константы имеет вид

$$
\ell(\mathbf{w}) = \log p(\mathbf{w} \mid \mathbf{X}, \mathbf{y}) = \sum_{i=1}^n \bigl[ y_i \log \sigma_i + (1-y_i)\log(1-\sigma_i) \bigr] - \frac{1}{2\sigma_w^2}\|\mathbf{w}\|^2 + \text{const},
\tag{1}
$$

где $\sigma_i = \sigma(\mathbf{x}_i^T \mathbf{w}) = 1/(1 + e^{-\mathbf{x}_i^T \mathbf{w}})$. Градиент $\ell(\mathbf{w})$ по $\mathbf{w}$:

$$
\nabla \ell(\mathbf{w}) = \sum_{i=1}^n (y_i - \sigma_i)\mathbf{x}_i - \frac{1}{\sigma_w^2}\mathbf{w}.
\tag{2}
$$

В векторно-матричной форме, обозначив $\boldsymbol{\sigma} = (\sigma_1,\dots,\sigma_n)^T$ и $\mathbf{X}$ — матрицу плана $n\times d$, получаем

$$
\nabla \ell(\mathbf{w}) = \mathbf{X}^T (\mathbf{y} - \boldsymbol{\sigma}) - \frac{1}{\sigma_w^2}\mathbf{w}.
\tag{3}
$$

#### 2. Метод Ньютона–Рафсона для поиска MAP

Максимум $\ell(\mathbf{w})$ находится приравниванием градиента к нулю. Получающаяся система нелинейна, и стандартным методом её решения является **метод Ньютона–Рафсона** (или просто метод Ньютона). Итерационное правило обновления весов:

$$
\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \bigl[\nabla^2 \ell(\mathbf{w}^{(t)})\bigr]^{-1} \nabla \ell(\mathbf{w}^{(t)}).
\tag{4}
$$

Гессиан $\nabla^2 \ell(\mathbf{w})$ (матрица вторых производных) для регуляризованной логистической регрессии был вычислен ранее:

$$
\nabla^2 \ell(\mathbf{w}) = -\mathbf{X}^T \mathbf{D} \mathbf{X} - \frac{1}{\sigma_w^2} \mathbf{I}_d,
\tag{5}
$$

где $\mathbf{D} = \operatorname{diag}(\sigma_1(1-\sigma_1), \dots, \sigma_n(1-\sigma_n))$. Подставляя (3) и (5) в (4) и меняя знак с учётом того, что и градиент, и гессиан взяты от логарифма апостериорной плотности (который максимизируется), получаем

$$
\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} + \bigl( \mathbf{X}^T \mathbf{D}^{(t)} \mathbf{X} + \tfrac{1}{\sigma_w^2} \mathbf{I} \bigr)^{-1} \bigl( \mathbf{X}^T (\mathbf{y} - \boldsymbol{\sigma}^{(t)}) - \tfrac{1}{\sigma_w^2} \mathbf{w}^{(t)} \bigr).
\tag{6}
$$

Это и есть итерационная схема Ньютона–Рафсона. Чтобы выявить её связь с IRLS, перепишем (6) в эквивалентной форме, подчеркнув сходство с решением взвешенной линейной регрессии. Введём **рабочий отклик** (adjusted response)

$$
\mathbf{z}^{(t)} = \mathbf{X} \mathbf{w}^{(t)} + \bigl(\mathbf{D}^{(t)}\bigr)^{-1} (\mathbf{y} - \boldsymbol{\sigma}^{(t)}),
$$

и диагональную весовую матрицу $\mathbf{W}^{(t)} = \mathbf{D}^{(t)}$. Тогда после алгебраических преобразований обновление (6) принимает вид нормальных уравнений взвешенного МНК с регуляризацией:

$$
\mathbf{w}^{(t+1)} = \bigl( \mathbf{X}^T \mathbf{W}^{(t)} \mathbf{X} + \tfrac{1}{\sigma_w^2} \mathbf{I} \bigr)^{-1} \mathbf{X}^T \mathbf{W}^{(t)} \mathbf{z}^{(t)}.
\tag{7}
$$

Таким образом, каждая итерация Ньютона эквивалентна решению задачи гребневой регрессии с весами $w_i = \sigma_i(1-\sigma_i)$ и целевой переменной $\mathbf{z}^{(t)}$. Отсюда и название — **итеративно перевзвешенный метод наименьших квадратов** (IRLS). Заметим, что при $\sigma_w^2 \to \infty$ регуляризация исчезает, и мы получаем классический IRLS для обычной (нерегуляризованной) логистической регрессии.

#### 3. Апостериорная ковариация

После того как итерации сошлись к $\mathbf{w}_{\text{MAP}}$, лапласовское приближение предписывает взять ковариационную матрицу апостериорного распределения как обратный гессиан в точке моды:

$$
\boldsymbol{\Sigma}_{\text{post}} = \bigl( -\nabla^2 \ell(\mathbf{w}_{\text{MAP}}) \bigr)^{-1} = \bigl( \mathbf{X}^T \mathbf{D}_{\text{MAP}} \mathbf{X} + \tfrac{1}{\sigma_w^2} \mathbf{I} \bigr)^{-1}.
\tag{8}
$$

Здесь $\mathbf{D}_{\text{MAP}}$ вычисляется с использованием $\boldsymbol{\sigma} = \sigma(\mathbf{X}\mathbf{w}_{\text{MAP}})$. Эта матрица играет роль апостериорной ковариации: $\mathbf{w} \mid \mathbf{X}, \mathbf{y} \approx \mathcal{N}(\mathbf{w}_{\text{MAP}}, \boldsymbol{\Sigma}_{\text{post}})$. Диагональные элементы $\boldsymbol{\Sigma}_{\text{post}}$ дают оценки апостериорной дисперсии каждого коэффициента, а внедиагональные — их ковариации. На основе этих величин строятся байесовские доверительные интервалы (credible intervals) и вычисляется неопределённость предсказаний.

Стоит отметить, что ковариационная матрица зависит от $\sigma_w^2$: чем меньше априорная дисперсия, тем сильнее регуляризация и тем меньше апостериорная неопределённость. При $\sigma_w^2 \to \infty$ ковариация стремится к $(\mathbf{X}^T \mathbf{D}_{\text{MAP}} \mathbf{X})^{-1}$, что является аналогом асимптотической ковариационной матрицы MLE.

#### 4. Роль априорной дисперсии $\sigma_w^2$

Параметр $\sigma_w^2$ играет двойственную роль. С одной стороны, он определяет априорную неопределённость весов и, следовательно, ширину апостериорного распределения. С другой стороны, в итерационной формуле (6)–(7) слагаемое $\frac{1}{\sigma_w^2} \mathbf{I}$ гарантирует, что матрица, подлежащая обращению, всегда положительно определена и хорошо обусловлена. Это чрезвычайно важно в следующих ситуациях:

- **Мультиколлинеарность признаков** — матрица $\mathbf{X}^T \mathbf{D} \mathbf{X}$ может быть вырожденной или плохо обусловленной; добавление $\frac{1}{\sigma_w^2} \mathbf{I}$ решает проблему.
- **Полная разделимость классов** — в отсутствие регуляризации метод Ньютона расходится, так как максимум правдоподобия достигается на бесконечности. Конечное $\sigma_w^2$ удерживает веса в конечной области, делая MAP-оценку единственной и численно устойчивой.

Таким образом, байесовский подход даже при использовании лишь точечной MAP-оценки включает естественную регуляризацию, которая делает задачу корректно поставленной.

#### 5. Углублённый взгляд: сходимость метода Ньютона и поведение при разделимости

Метод Ньютона обладает квадратичной сходимостью вблизи оптимума, что делает его значительно быстрее градиентного спуска. Однако для логистической регрессии без регуляризации он может расходиться, если данные допускают полную (или квази-полную) разделимость. Формально, разделимость означает существование вектора $\mathbf{w}$ такого, что $\mathbf{x}_i^T \mathbf{w} > 0$ для всех $i$ с $y_i = 1$ и $\mathbf{x}_i^T \mathbf{w} < 0$ для всех $i$ с $y_i = 0$. В этом случае логарифмическое правдоподобие не ограничено сверху, и максимум достигается при $\|\mathbf{w}\| \to \infty$. Гессиан при этом вырождается (веса $w_i = \sigma_i(1-\sigma_i)$ стремятся к нулю), и метод Ньютона становится численно неустойчивым или расходится.

Добавление регуляризационного члена $\frac{1}{\sigma_w^2} \mathbf{I}$ в гессиан полностью устраняет эту проблему: даже если данные разделимы, апостериорное распределение остаётся собственным, а его мода конечна. Итерации Ньютона с регуляризацией сходятся к единственному решению, поскольку целевая функция строго вогнута (гессиан отрицательно определён). На практике обычно используют умеренное значение $\sigma_w^2$ (или, что эквивалентно, настраивают параметр регуляризации $\lambda = 1/\sigma_w^2$) и применяют защитные меры, такие как остановка при малом изменении весов или применении пошагового подбора длины шага.

Таким образом, метод Ньютона–Рафсона в форме IRLS не только эффективно находит MAP-оценку для байесовской логистической регрессии, но и попутно выдаёт матрицу, обратную апостериорной ковариации. Это делает лапласовское приближение чрезвычайно удобным вычислительным инструментом для получения как точечных, так и интервальных выводов.

---

#### Вопросы для самопроверки

**Для начинающих**
1. Как выглядит градиент логарифма апостериорной плотности в логистической регрессии? Почему он включает разность наблюдаемых и предсказанных вероятностей?
2. Опишите идею метода Ньютона–Рафсона. Почему он сходится быстрее градиентного спуска?
3. Как из матрицы, обращённой на последней итерации Ньютона, получить апостериорную ковариационную матрицу? Что означают её диагональные элементы?
4. Зачем в байесовской логистической регрессии нужна регуляризация? Какое слагаемое в формулах (6)–(8) за неё отвечает?
5. Почему при полной разделимости классов нерегуляризованный метод Ньютона может расходиться?

**Для профессионалов**
1. Выведите итерационную формулу IRLS (7) из общего обновления Ньютона (6). Покажите, что она эквивалентна решению взвешенной гребневой регрессии с рабочим откликом $\mathbf{z}$.
2. Докажите, что матрица $\mathbf{X}^T \mathbf{D} \mathbf{X}$ положительно полуопределена, и объясните, почему добавление $\frac{1}{\sigma_w^2}\mathbf{I}$ делает её положительно определённой.
3. Сравните скорость сходимости метода Ньютона и стохастического градиентного спуска (SGD) для логистической регрессии. В каких случаях предпочтительнее SGD?
4. При больших $n$ прямое обращение $d\times d$ матрицы может быть дорогим. Опишите, как можно решать ньютоновские уравнения с помощью метода сопряжённых градиентов (CG), не формируя матрицу явно.
5. Как модифицировать алгоритм для использования квазиньютоновского метода L-BFGS вместо точного Ньютона? Какие преимущества это даёт в высокоразмерных задачах?

### Предсказательное распределение в байесовской логистической регрессии

Лапласовское приближение даёт нам удобную гауссовскую аппроксимацию апостериорного распределения весов $p(\mathbf{w} \mid \mathbf{X}, \mathbf{y}) \approx \mathcal{N}(\mathbf{w} \mid \mathbf{w}_{\text{MAP}}, \boldsymbol{\Sigma}_{\text{post}})$. Однако конечная цель классификации — не сама оценка $\mathbf{w}$, а предсказание вероятности $y_* = 1$ для нового объекта $\mathbf{x}_*$. Точечная подстановка $\mathbf{w}_{\text{MAP}}$ в сигмоиду игнорирует апостериорную неопределённость весов и, как правило, даёт чрезмерно уверенные, плохо откалиброванные вероятности. Правильный байесовский подход требует проинтегрировать сигмоидальную функцию по апостериорному распределению.

#### 1. Интеграл для предсказательной вероятности

Искомая предсказательная вероятность класса 1 для нового вектора признаков $\mathbf{x}_*$ определяется как математическое ожидание $\sigma(\mathbf{x}_*^T \mathbf{w})$ по апостериорному распределению:

$$
p(y_*=1 \mid \mathbf{x}_*, \mathbf{X},\mathbf{y}) = \int \sigma(\mathbf{x}_*^T \mathbf{w}) \; p(\mathbf{w} \mid \mathbf{X},\mathbf{y}) \, d\mathbf{w}.
\tag{1}
$$

Даже при гауссовской аппроксимации $p(\mathbf{w} \mid \mathbf{X},\mathbf{y}) \approx \mathcal{N}(\mathbf{w} \mid \boldsymbol{\mu}_{\text{post}}, \boldsymbol{\Sigma}_{\text{post}})$ этот интеграл не выражается в элементарных функциях, поскольку свёртка логистической функции с нормальной плотностью не имеет замкнутого вида. Поэтому применяются различные приближения.

#### 2. Аппроксимация через пробит-функцию

Наиболее популярный аналитический метод опирается на близость логистической функции $\sigma(z)$ к функции распределения стандартного нормального закона $\Phi(z)$. Хорошо известно, что

$$
\sigma(z) \approx \Phi(\kappa z), \qquad \kappa = \sqrt{\frac{\pi}{8}} \approx 0.626.
\tag{2}
$$

Максимальная абсолютная ошибка этого приближения не превышает 0.01, и оно широко используется в байесовской статистике. Главное достоинство замены в том, что интеграл от $\Phi(\mathbf{a}^T \mathbf{w})$ по гауссовскому распределению $\mathbf{w}$ вычисляется аналитически. Если $\mathbf{w} \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$, то линейная комбинация $a = \mathbf{x}_*^T \mathbf{w}$ также нормальна: $a \sim \mathcal{N}(\mathbf{x}_*^T \boldsymbol{\mu}, \; \mathbf{x}_*^T \boldsymbol{\Sigma} \,\mathbf{x}_*)$. Тогда, представляя $\Phi$ как вероятность $P(Z \le a)$ с независимой $Z \sim \mathcal{N}(0,1)$, получаем

$$
\begin{aligned}
\int \Phi(\mathbf{x}_*^T \mathbf{w}) \, \mathcal{N}(\mathbf{w} \mid \boldsymbol{\mu},\boldsymbol{\Sigma}) \, d\mathbf{w}
&= \mathbb{P}\bigl( Z \le \mathbf{x}_*^T \mathbf{w} \bigr) \\
&= \Phi\!\left( \frac{\mathbf{x}_*^T \boldsymbol{\mu}}{\sqrt{1 + \mathbf{x}_*^T \boldsymbol{\Sigma} \,\mathbf{x}_*}} \right).
\end{aligned}
\tag{3}
$$

Возвращаясь от $\Phi$ к $\sigma$ посредством обратного преобразования $\Phi(\kappa^{-1} \cdot)$, получаем окончательную формулу приближённой предсказательной вероятности:

$$
\boxed{ p(y_*=1 \mid \mathbf{x}_*, \mathbf{X},\mathbf{y}) \;\approx\; \sigma\!\left( \frac{\mathbf{x}_*^T \boldsymbol{\mu}_{\text{post}}}{\sqrt{1 + \frac{\pi}{8}\,\mathbf{x}_*^T \boldsymbol{\Sigma}_{\text{post}} \,\mathbf{x}_*}} \right) }.
\tag{4}
$$

Эта формула, известная как приближение МакКаллага (или «Mackay’s method»), имеет прозрачную структуру: знаменатель уменьшает модуль аргумента сигмоиды, сдвигая предсказанную вероятность к $0.5$ в областях, где апостериорная неопределённость $\mathbf{x}_*^T \boldsymbol{\Sigma}_{\text{post}} \mathbf{x}_*$ велика. Тем самым модель автоматически становится более осторожной вдали от обучающих данных.

#### 3. Метод Монте-Карло

Прямолинейный и вместе с тем наиболее точный (в пределе большого числа сэмплов) способ вычисления интеграла (1) — метод Монте-Карло. Из аппроксимирующего гауссовского распределения генерируется $S$ независимых реализаций весов

$$
\mathbf{w}_s \sim \mathcal{N}\bigl(\boldsymbol{\mu}_{\text{post}}, \boldsymbol{\Sigma}_{\text{post}}\bigr), \quad s=1,\dots,S,
$$

и предсказательная вероятность оценивается эмпирическим средним:

$$
\hat{p}(y_*=1 \mid \mathbf{x}_*, \mathbf{X},\mathbf{y}) = \frac{1}{S} \sum_{s=1}^S \sigma(\mathbf{x}_*^T \mathbf{w}_s).
\tag{5}
$$

При $S \to \infty$ эта оценка сходится к точному значению интеграла (1) для заданной гауссовской аппроксимации. На практике уже $S \approx 1000$ даёт хорошую точность. Метод не требует аналитических приближений, легко реализуется и обобщается на произвольные модели. Недостатком является вычислительная стоимость: для каждого предсказания необходимо $S$ раз вычислить $\mathbf{x}_*^T \mathbf{w}_s$ и применить сигмоиду. Это может быть критично в приложениях реального времени, но вполне приемлемо при пакетной обработке или умеренном числе тестовых точек.

Метод Монте-Карло также применим непосредственно к точному апостериорному распределению, полученному MCMC-сэмплированием (например, HMC). В этом случае он даёт состоятельную оценку настоящего байесовского предсказания без каких-либо аппроксимаций.

#### 4. Сравнение приближений

Полезно сопоставить три подхода к предсказанию:

- **Точечный прогноз по MAP** — $\sigma(\mathbf{x}_*^T \mathbf{w}_{\text{MAP}})$ — полностью игнорирует неопределённость весов, давая чрезмерно экстремальные вероятности (близкие к $0$ или $1$), и имеет плохую калибровку.
- **Пробит-приближение** (4) — аналитически учитывает апостериорную дисперсию линейного предиктора и сдвигает вероятности к $0.5$ пропорционально неопределённости. Оно вычислительно дёшево и часто даёт хорошо откалиброванные предсказания.
- **Метод Монте-Карло** — более точная численная оценка интеграла (1) в рамках заданной гауссовской аппроксимации. Различия с пробит-приближением обычно невелики, но MC может быть предпочтительнее, когда форма апостериорного распределения сильно негауссовская (хотя в лапласовском приближении мы его как раз считаем гауссовским).

> **Углублённый взгляд: сравнение точности и связь с латентной переменной**  
> Пробит-аппроксимация является не просто эвристикой: она представляет собой точное решение для модели, в которой сигмоида заменена на пробит-функцию. Действительно, рассмотрим модель с латентной переменной $t = \mathbf{x}^T \mathbf{w} + \varepsilon$, где $\varepsilon \sim \mathcal{N}(0,1)$, а наблюдение полагается равным $y=1$ при $t > 0$. Тогда $P(y=1 \mid \mathbf{x}, \mathbf{w}) = \Phi(\mathbf{x}^T \mathbf{w})$. При априорном $\mathbf{w} \sim \mathcal{N}(\boldsymbol{\mu}_0, \boldsymbol{\Sigma}_0)$ апостериорное распределение $\mathbf{w}$ остаётся гауссовским (приближённо), а предсказательная вероятность для новой точки даётся формулой (3) без дополнительных констант. Замена $\Phi$ на $\sigma$ с масштабирующим фактором $\kappa$ делает эту модель почти эквивалентной логистической регрессии, одновременно сохраняя аналитическую интегрируемость.  
> Сравнение численных значений: для широкого диапазона параметров пробит-приближение и Монте-Карло с $S=10^4$ дают практически совпадающие результаты, тогда как наивная подстановка MAP систематически занижает энтропию предсказаний. Таким образом, на практике пробит-приближение (4) является рекомендуемым методом по умолчанию, а MC используется либо для проверки, либо в тех редких случаях, когда требуется более высокая точность для конкретной формы распределения.

---

#### Вопросы для самопроверки

**Для начинающих**
1. Почему нельзя просто вычислить вероятность класса, подставив MAP-оценку весов в сигмоиду?
2. Зачем при байесовском предсказании необходимо вычислять интеграл по апостериорному распределению весов?
3. Что такое пробит-аппроксимация и как она помогает получить замкнутую формулу для предсказания?
4. Как работает метод Монте-Карло для оценки предсказательной вероятности? Сколько сэмплов обычно требуется?
5. Какой из методов (пробит или Монте-Карло) является более точным, и в каких случаях разница между ними существенна?

**Для профессионалов**
1. Выведите формулу $\int \Phi(\mathbf{a}^T \mathbf{w}) \mathcal{N}(\mathbf{w} \mid \boldsymbol{\mu}, \boldsymbol{\Sigma}) d\mathbf{w} = \Phi\!\left( \frac{\mathbf{a}^T \boldsymbol{\mu}}{\sqrt{1 + \mathbf{a}^T \boldsymbol{\Sigma} \mathbf{a}}} \right)$, используя свойства многомерного нормального распределения.
2. Объясните, почему замена $\sigma(z) \approx \Phi(\kappa z)$ с $\kappa = \sqrt{\pi/8}$ даёт хорошее приближение для интеграла свёртки, и как коэффициент $\kappa$ связан с дисперсиями логистического и нормального распределений.
3. Сравните вычислительную сложность получения предсказательной вероятности с помощью пробит-приближения и метода Монте-Карло для одного тестового объекта. При каком количестве тестовых точек MC становится неприемлемым?
4. Покажите, что пробит-приближение может быть интерпретировано как точное предсказание для модели с латентной нормальной переменной, и обсудите, как изменится вывод апостериорного распределения в такой модели.

### Реализация байесовской логистической регрессии на Python (MAP и лапласовское приближение)

В предыдущих разделах мы вывели математические основы байесовской логистической регрессии: метод Ньютона–Рафсона (IRLS) для поиска MAP-оценки, вычисление апостериорной ковариации и приближённое предсказательное распределение через пробит-аппроксимацию. Теперь мы воплотим эти идеи в программном коде, создав класс `BayesianLogisticRegressionLaplace`. Он будет находить моду апостериорного распределения, вычислять её ковариацию и предсказывать вероятности с учётом неопределённости. Затем сравним модель с классической логистической регрессией и проиллюстрируем байесовские доверительные интервалы.

В дальнейшем предполагается, что необходимые библиотеки импортированы:

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve
from scipy.stats import norm
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
from sklearn.calibration import calibration_curve
```

#### 1. Реализация IRLS для MAP и апостериорной ковариации

Класс `BayesianLogisticRegressionLaplace` принимает параметр `alpha = 1/σ_w²` — точность априорного распределения. Метод `fit` реализует итерации Ньютона–Рафсона для максимизации логарифма апостериорной плотности. На каждом шаге вычисляются:

- предсказанные вероятности $\boldsymbol{\sigma} = \sigma(\mathbf{X}\mathbf{w})$,
- диагональная матрица $\mathbf{D} = \operatorname{diag}(\sigma_i(1-\sigma_i))$,
- гессиан $\mathbf{H} = \mathbf{X}^T \mathbf{D} \mathbf{X} + \alpha \mathbf{I}$,
- градиент $\mathbf{g} = \mathbf{X}^T (\mathbf{y} - \boldsymbol{\sigma}) - \alpha \mathbf{w}$,
- обновление $\mathbf{w}_{\text{new}} = \mathbf{w} + \mathbf{H}^{-1} \mathbf{g}$ (эквивалентно решению $\mathbf{H} \mathbf{w}_{\text{new}} = \mathbf{X}^T \mathbf{W} \mathbf{z}$, но мы используем прямой способ).

После сходимости сохраняется итоговый вектор весов `self.w_` и апостериорная ковариация `self.cov_ = inv(H)`.

```python
class BayesianLogisticRegressionLaplace:
    def __init__(self, alpha=1.0):
        """
        alpha: точность априорного распределения (1/σ_w^2).
               Чем больше alpha, тем сильнее регуляризация.
        """
        self.alpha = alpha
        self.w_ = None
        self.cov_ = None

    @staticmethod
    def sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))  # предотвращаем переполнение

    def fit(self, X, y, max_iter=100, tol=1e-6):
        n, d = X.shape
        # Инициализация нулевым вектором
        w = np.zeros(d)
        for i in range(max_iter):
            p = self.sigmoid(X @ w)
            D = np.diag(p * (1 - p))
            H = X.T @ D @ X + self.alpha * np.eye(d)
            g = X.T @ (y - p) - self.alpha * w
            # Решаем H Δw = g  (поскольку обновление w_new = w + H^{-1} g)
            try:
                delta = solve(H, g, assume_a='pos')
            except np.linalg.LinAlgError:
                # Если матрица близка к вырожденной, добавляем немного регуляризации
                delta = solve(H + 1e-6 * np.eye(d), g, assume_a='pos')
            w_new = w + delta
            # Проверка сходимости
            if np.linalg.norm(delta) < tol:
                w = w_new
                break
            w = w_new
        self.w_ = w
        # Апостериорная ковариация: обратный гессиан в найденной MAP-точке
        p = self.sigmoid(X @ w)
        D = np.diag(p * (1 - p))
        H = X.T @ D @ X + self.alpha * np.eye(d)
        self.cov_ = np.linalg.inv(H)
        return self
```

Несколько замечаний:
- Сигмоида обрезается по входу для численной устойчивости при больших по модулю значениях.
- Гессиан всегда положительно определён благодаря добавлению $\alpha \mathbf{I}$, что гарантирует корректность лапласовского приближения и единственность MAP-оценки (если данные не полностью разделимы).
- При полной разделимости $\alpha$ удерживает веса конечными, но может потребоваться более сильная регуляризация для сходимости; на практике умеренные $\alpha$ работают хорошо.

#### 2. Предсказательные вероятности с учётом неопределённости

Метод `predict_proba` возвращает вероятность класса 1. Реализуем два подхода: пробит-аппроксимацию (основной, быстрый) и сэмплирование Монте-Карло (для проверки и более точных оценок). Для пробит-аппроксимации используем формулу

$$
p(y=1 \mid \mathbf{x}_*) \approx \sigma\left(\frac{\mu_*}{\sqrt{1 + \frac{\pi}{8} s_*^2}}\right),
$$

где $\mu_* = \mathbf{x}_*^T \mathbf{w}_{\text{MAP}}$, $s_*^2 = \mathbf{x}_*^T \boldsymbol{\Sigma}_{\text{post}} \mathbf{x}_*$.

```python
    def predict_proba(self, X, method='probit', n_samples=500):
        if self.w_ is None:
            raise RuntimeError("Модель не обучена. Вызовите fit().")
        mu = X @ self.w_
        if method == 'probit':
            # Диагональ X @ cov @ X^T
            var = np.sum(X @ self.cov_ * X, axis=1)
            kappa = np.sqrt(1.0 + np.pi * var / 8.0)
            prob = self.sigmoid(mu / kappa)
            return prob
        elif method == 'mc':
            # Сэмплируем из апостериорного распределения весов
            w_samples = np.random.multivariate_normal(self.w_, self.cov_, size=n_samples)
            # Вычисляем сигмоиду для каждого сэмпла и усредняем
            prob = np.mean(self.sigmoid(X @ w_samples.T), axis=1)
            return prob
        else:
            raise ValueError("method должен быть 'probit' или 'mc'")

    def predict(self, X, threshold=0.5, **kwargs):
        proba = self.predict_proba(X, **kwargs)
        return (proba >= threshold).astype(int)
```

#### 3. Сравнение с обычной логистической регрессией

Протестируем нашу модель на наборе данных `breast_cancer` и сравним с `LogisticRegression` из scikit-learn (без регуляризации, т.е. `C` очень большое). Оценим точность, AUC и калибровку с помощью кривой надёжности (reliability curve) и индекса Бриера.

```python
data = load_breast_cancer()
X, y = data.data, data.target
# Масштабируем признаки для лучшей сходимости
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# Байесовская модель с умеренной регуляризацией
blr = BayesianLogisticRegressionLaplace(alpha=1.0)
blr.fit(X_train, y_train)
p_pred_bayes = blr.predict_proba(X_test, method='probit')
y_pred_bayes = blr.predict(X_test)

# Обычная логистическая регрессия (sklearn)
lr = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
lr.fit(X_train, y_train)
p_pred_lr = lr.predict_proba(X_test)[:, 1]
y_pred_lr = lr.predict(X_test)

print("Байесовская LR (probit) accuracy:", accuracy_score(y_test, y_pred_bayes))
print("sklearn LR accuracy:", accuracy_score(y_test, y_pred_lr))
print("Байесовская LR AUC:", roc_auc_score(y_test, p_pred_bayes))
print("sklearn LR AUC:", roc_auc_score(y_test, p_pred_lr))
print("Байесовская LR Brier:", brier_score_loss(y_test, p_pred_bayes))
print("sklearn LR Brier:", brier_score_loss(y_test, p_pred_lr))
```

Обычно точность и AUC практически совпадают, но байесовская модель даёт чуть менее уверенные (и лучше откалиброванные) вероятности при сильной регуляризации. Построим калибровочные кривые.

```python
plt.figure(figsize=(8, 6))
for probs, label in [(p_pred_bayes, 'Bayesian LR (probit)'), (p_pred_lr, 'Logistic Regression')]:
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_test, probs, n_bins=10, strategy='uniform')
    plt.plot(mean_predicted_value, fraction_of_positives, 's-', label=label)
plt.plot([0, 1], [0, 1], 'k:', label='Perfect calibration')
plt.xlabel('Средняя предсказанная вероятность')
plt.ylabel('Доля положительных примеров')
plt.title('Калибровочные кривые')
plt.legend()
plt.grid(True)
plt.show()
```

Байесовская кривая, как правило, лежит ближе к диагонали, особенно при малых выборках, демонстрируя преимущество учёта неопределённости весов.

#### 4. Визуализация байесовских доверительных интервалов для вероятности

Для наглядности сгенерируем одномерные данные: $p(y=1 \mid x) = \sigma(2x - 1)$ с шумом. Обучим нашу модель и построим график предсказанной вероятности вместе с 95% доверительным интервалом, полученным через дельта-метод: берём интервал для линейного предиктора $\mu_* \pm 1.96 s_*$ и преобразуем через сигмоиду.

```python
np.random.seed(42)
n = 80
x = np.random.uniform(-3, 3, n)
a_true = 2 * x - 1
p_true = 1 / (1 + np.exp(-a_true))
y = (np.random.rand(n) < p_true).astype(int)
X = x.reshape(-1, 1)

# Обучаем модель
blr_1d = BayesianLogisticRegressionLaplace(alpha=1.0)
blr_1d.fit(X, y)

# Сетка для визуализации
x_grid = np.linspace(-3.5, 3.5, 200).reshape(-1, 1)
mu_grid = x_grid @ blr_1d.w_
# Дисперсия линейного предиктора
s2_grid = np.sum(x_grid @ blr_1d.cov_ * x_grid, axis=1)
s_grid = np.sqrt(s2_grid)
# Предсказанная вероятность (probit)
p_grid = blr_1d.predict_proba(x_grid, method='probit')
# 95% доверительный интервал для линейного предиктора
a_low = mu_grid - 1.96 * s_grid
a_high = mu_grid + 1.96 * s_grid
# Преобразуем границы в вероятности
p_low = 1 / (1 + np.exp(-a_low))
p_high = 1 / (1 + np.exp(-a_high))

plt.figure(figsize=(9, 6))
plt.scatter(x, y, c=y, cmap='bwr', edgecolor='k', s=50, label='Данные')
plt.plot(x_grid, p_true, 'g--', label='Истинная вероятность')
plt.plot(x_grid, p_grid, 'r-', label='Байесовское среднее (probit)')
plt.fill_between(x_grid.ravel(), p_low, p_high, color='red', alpha=0.15,
                 label='95% байесовский интервал')
plt.xlabel('x')
plt.ylabel('P(y=1 | x)')
plt.title('Байесовская логистическая регрессия с лапласовским приближением')
plt.legend()
plt.grid(True)
plt.show()
```

Интервал расширяется на краях, где данных мало, что отражает рост эпистемической неопределённости — свойство, недоступное точечной логистической регрессии.

#### 5. Заключение

Мы реализовали полноценный модуль байесовской логистической регрессии на основе лапласовского приближения. Метод Ньютона–Рафсона (IRLS) эффективно находит MAP-оценку, а обратный гессиан даёт апостериорную ковариацию. Предсказательные вероятности, вычисленные с помощью пробит-аппроксимации или Монте-Карло, учитывают неопределённость весов и обычно лучше откалиброваны, чем точечные аналоги. Полученный класс легко расширить на многоклассовый случай (one-vs-rest или мультиномиальный логит с аппроксимацией), а также на использование априорного среднего $\mathbf{m}_0 \neq \mathbf{0}$.

Эта реализация служит отличной основой для дальнейших байесовских методов — вариационного вывода, MCMC или иерархических моделей.

---

#### Вопросы для самопроверки

**Для начинающих**
1. Как в методе `fit` вычисляется гессиан $\mathbf{H}$ и для чего он используется в методе Ньютона?
2. Почему в формуле обновления весов мы решаем систему $\mathbf{H} \Delta \mathbf{w} = \mathbf{g}$, а не вычисляем $\mathbf{H}^{-1}$ напрямую?
3. В чём разница между `predict_proba` с методом `'probit'` и `'mc'`?
4. Как в коде получают 95% доверительный интервал для вероятности? Почему он расширяется на краях области признаков?

**Для профессионалов**
1. Реализуйте проверку на разделимость данных: если на какой-либо итерации веса неограниченно растут (например, $\|\mathbf{w}\| > 10^3$), то выдавайте предупреждение или автоматически увеличивайте `alpha`.
2. Сравните скорость сходимости реализованного IRLS и метода L-BFGS из `scipy.optimize.minimize` при большом числе признаков ($d \approx 500$). В каких случаях IRLS предпочтительнее?
3. Модифицируйте класс, добавив возможность задания априорного среднего `m0`. Выведите соответствующее изменение в градиенте и гессиане.
4. Как с помощью дельта-метода получить доверительный интервал для вероятности, аналогичный показанному в визуализации, без явного вычисления $s^2$? Покажите, что асимптотически он совпадает с интервалом из пробит-приближения.
5. Предложите способ расширения модели на случай многоклассовой логистической регрессии (softmax) с лапласовской аппроксимацией. Какие изменения потребуются в вычислении гессиана?

### Иерархические байесовские модели, робастные априорные распределения и сравнение с частотными методами

До сих пор мы рассматривали байесовскую логистическую регрессию с фиксированным априорным распределением $\mathbf{w} \sim \mathcal{N}(\mathbf{0}, \sigma_w^2 \mathbf{I})$ и использовали лапласовское приближение для вывода. Этот подход уже дал нам регуляризованную точечную оценку и гауссовскую аппроксимацию апостериорной неопределённости. Однако полная сила байесовского подхода раскрывается, когда мы позволяем данным влиять на силу регуляризации через иерархические априорные распределения и когда мы заменяем гауссовское априорное на более гибкие распределения с тяжёлыми хвостами, обеспечивающие робастность и разреженность. В этом разделе мы строго выведем соответствующие модели, обсудим их связь с частотными аналогами и продемонстрируем современные методы вывода с помощью MCMC.

#### 1. Иерархические априорные распределения (гиперприоры)

В предыдущих разделах априорная дисперсия $\sigma_w^2$ считалась известной константой. В иерархической байесовской модели она сама наделяется априорным распределением — гиперприором. Рассмотрим трёхуровневую модель

$$
\begin{aligned}
y_i \mid \mathbf{x}_i, \mathbf{w} &\sim \operatorname{Bernoulli}\bigl(\sigma(\mathbf{x}_i^T \mathbf{w})\bigr), \quad i=1,\dots,n, \\[4pt]
\mathbf{w} \mid \sigma_w^2 &\sim \mathcal{N}(\mathbf{0}, \sigma_w^2 \mathbf{I}_d), \\[4pt]
\sigma_w^2 &\sim p(\sigma_w^2).
\end{aligned}
$$

Совместное апостериорное распределение параметров $(\mathbf{w}, \sigma_w^2)$ с точностью до нормировочной константы имеет вид

$$
p(\mathbf{w}, \sigma_w^2 \mid \mathbf{X}, \mathbf{y}) \propto
\prod_{i=1}^n \sigma(\mathbf{x}_i^T \mathbf{w})^{y_i} \bigl(1 - \sigma(\mathbf{x}_i^T \mathbf{w})\bigr)^{1 - y_i}
\;\cdot\;
\mathcal{N}(\mathbf{w} \mid \mathbf{0}, \sigma_w^2 \mathbf{I}_d)
\;\cdot\;
p(\sigma_w^2). \tag{1}
$$

Наиболее распространённые гиперприоры:

- **Обратное гамма-распределение**: $\sigma_w^2 \sim \operatorname{Inv-Gamma}(a, b)$, т.е.
  $$
  p(\sigma_w^2) = \frac{b^a}{\Gamma(a)} (\sigma_w^2)^{-a-1} \exp\!\Bigl(-\frac{b}{\sigma_w^2}\Bigr), \quad \sigma_w^2 > 0.
  $$
- **Полукоши на шкалу** $\sigma_w$: $\sigma_w \sim \operatorname{Cauchy}^+(0, \beta)$,
  $$
  p(\sigma_w) = \frac{2}{\pi\beta \bigl(1 + (\sigma_w / \beta)^2\bigr)}, \quad \sigma_w \ge 0.
  $$

Выбор полукоши часто предпочтительнее, так как её тяжёлые хвосты позволяют апостериорному распределению $\sigma_w$ принимать большие значения при наличии сильного сигнала, не подавляя его излишней регуляризацией, и в то же время сильно сжимать веса при слабом сигнале.

Интегрируя $\sigma_w^2$ из (1), можно получить маргинальное априорное распределение весов

$$
p(\mathbf{w}) = \int_0^\infty \mathcal{N}(\mathbf{w} \mid \mathbf{0}, \sigma_w^2 \mathbf{I}_d) \; p(\sigma_w^2) \, d\sigma_w^2.
$$

Для обратного гамма-гиперприора это приводит к многомерному $t$-распределению Стьюдента, что делает модель более робастной к выбросам в пространстве параметров.

**Вывод с помощью MCMC.** Прямое аналитическое вычисление апостериорного распределения (1) невозможно, однако современные алгоритмы MCMC, такие как гамильтонов Монте-Карло (HMC) и его адаптивная версия NUTS, позволяют эффективно генерировать выборку из $p(\mathbf{w}, \sigma_w^2 \mid \mathbf{X}, \mathbf{y})$. Вероятностные языки программирования (PyMC, Stan, NumPyro) автоматически строят граф модели и выполняют сэмплирование.

#### 2. Робастные априорные распределения

Гауссовское априорное распределение не всегда адекватно: оно не способно одновременно сохранять большие коэффициенты для релевантных признаков и сильно сжимать остальные. Эту проблему решают априорные распределения с тяжёлыми хвостами.

**Байесовский лассо (Laplace prior).** Положим независимое априорное распределение Лапласа на каждый вес:

$$
p(\mathbf{w} \mid \lambda) = \prod_{j=1}^d \frac{\lambda}{2} \exp\bigl(-\lambda |w_j|\bigr). \tag{2}
$$

Максимизация логарифма апостериорной плотности с таким априорным приводит к классическому лассо (L1-регуляризации). Однако полный байесовский вывод даёт апостериорное распределение весов, которое может концентрироваться около нуля, но при этом иметь тяжёлые хвосты, что даёт более реалистичную картину неопределённости.

Ключевой технический приём — представление распределения Лапласа в виде масштабной смеси нормальных распределений. Для каждого $w_j$ вводится латентная переменная $\tau_j^2 \sim \operatorname{Exponential}(\lambda^2/2)$:

$$
\frac{\lambda}{2} e^{-\lambda |w_j|} = \int_0^\infty \frac{1}{\sqrt{2\pi\tau_j^2}} \exp\!\Bigl(-\frac{w_j^2}{2\tau_j^2}\Bigr) \; \frac{\lambda^2}{2} e^{-\lambda^2 \tau_j^2 / 2} \, d\tau_j^2. \tag{3}
$$

Тем самым модель записывается как иерархическая нормальная:

$$
\begin{aligned}
y_i \mid \mathbf{x}_i, \mathbf{w} &\sim \operatorname{Bernoulli}\bigl(\sigma(\mathbf{x}_i^T \mathbf{w})\bigr), \\
\mathbf{w} \mid \{\tau_j^2\} &\sim \mathcal{N}\bigl(\mathbf{0}, \operatorname{diag}(\tau_1^2,\dots,\tau_d^2)\bigr), \\
\tau_j^2 \mid \lambda &\sim \operatorname{Exponential}(\lambda^2 / 2),
\end{aligned}
$$

что чрезвычайно удобно для MCMC-сэмплирования.

**Робастная логистическая регрессия с $t$-априорным.** Как отмечалось выше, выбор обратного гамма-гиперприора на $\sigma_w^2$ эквивалентен $t$-априорному на $\mathbf{w}$. Такие распределения подавляют влияние наблюдений с экстремальными значениями линейного предиктора, делая модель более устойчивой к выбросам.

#### 3. Сравнение с частотной логистической регрессией

Сведём ключевые различия в таблицу, а затем дадим математические обоснования.

| Критерий | Частотная логистическая регрессия | Байесовская логистическая регрессия |
|----------|-----------------------------------|--------------------------------------|
| Оценка параметров | Точечная (MLE или регуляризованный MLE) | Апостериорное распределение |
| Неопределённость | Асимптотические доверительные интервалы, bootstrap | Credible intervals из апостериорного (точные при любом $n$) |
| Регуляризация | Штрафной член, подбор $\lambda$ кросс-валидацией | Априорное распределение; возможно иерархическое |
| Мультиколлинеарность | Плохая обусловленность; Ridge/Lasso помогают | Гауссовское априорное = Ridge; Лаплас = разреженность |
| Полная разделимость | MLE не существует, нужна коррекция (Firth) или регуляризация | Априорное всегда даёт собственное апостериорное |
| Интерпретация | $p$-значения, тесты Вальда | Вероятностные утверждения (например, $P(w_j>0\mid \mathbf{y})$) |
| Сравнение моделей | AIC, кросс-валидация | WAIC, LOO-CV, байесовские факторы |

**Проблема разделимости: формальное доказательство.** В частотной логистической регрессии без регуляризации логарифмическое правдоподобие $\ell(\mathbf{w})$ неограниченно возрастает при полной разделимости, так как существует $\mathbf{u}$ такой, что $\mathbf{x}_i^T \mathbf{u} > 0$ для всех $i$ с $y_i=1$ и $\mathbf{x}_i^T \mathbf{u} < 0$ для $y_i=0$. Тогда для любого $c>0$ имеем $\ell(c\mathbf{u}) \to 0$ (потери стремятся к нулю), а $\|\mathbf{w}\| \to \infty$. Гессиан $\mathbf{H} = \mathbf{X}^T \mathbf{D} \mathbf{X}$ при этом становится вырожденным, итерации Ньютона расходятся. В байесовской же модели с собственным априорным $p(\mathbf{w})$ апостериорное распределение всегда собственное, так как

$$
\int p(\mathbf{y} \mid \mathbf{X}, \mathbf{w}) p(\mathbf{w}) d\mathbf{w} \le \int p(\mathbf{w}) d\mathbf{w} = 1,
$$

поэтому нормировочная константа конечна, а апостериорные моменты существуют. Следовательно, MAP-оценка конечна и единственна, а MCMC сходится.

**Сравнение интервалов.** Частотный $100(1-\alpha)\%$ доверительный интервал для $w_j$ имеет вид $\hat{w}_j \pm z_{1-\alpha/2} \sqrt{[\mathbf{I}(\hat{\mathbf{w}})^{-1}]_{jj}}$, где $\mathbf{I}$ — наблюдаемая информация Фишера. Байесовский credible interval из лапласовского приближения: $\mu_{\text{post},j} \pm z_{1-\alpha/2} \sqrt{\Sigma_{\text{post},jj}}$. При больших выборках они численно близки, но интерпретация разная. Для точного байесовского вывода credible interval строится по квантилям апостериорных сэмплов.

#### 4. Пример на реальных данных с PyMC

Приведём реализацию иерархической модели с априорным Лапласа (байесовский лассо) для логистической регрессии с помощью PyMC. Модель использует автоматическое сэмплирование NUTS.

```python
import numpy as np
import pymc as pm

# Генерация разреженных данных
np.random.seed(42)
n, d = 200, 20
X = np.random.randn(n, d)
true_w = np.zeros(d)
true_w[:5] = [2.0, -1.5, 1.0, -0.8, 0.5]
eta = X @ true_w
y = np.random.binomial(1, 1.0 / (1.0 + np.exp(-eta)))

# Модель
with pm.Model() as logistic_lasso:
    # Гиперприор на параметр масштаба Лапласа
    lam = pm.HalfCauchy('lam', beta=1.0)
    # Локальные дисперсии tau^2
    tau2 = pm.Exponential('tau2', lam=lam**2 / 2, shape=d)
    # Веса
    w = pm.Normal('w', mu=0, sigma=pm.math.sqrt(tau2), shape=d)
    # Линейный предиктор
    eta_obs = pm.math.dot(X, w)
    # Наблюдения
    y_obs = pm.Bernoulli('y_obs', logit_p=eta_obs, observed=y)
    
    # Сэмплирование
    trace = pm.sample(2000, tune=2000, target_accept=0.9, return_inferencedata=True)
```

После сэмплирования можно получить апостериорные распределения весов и предсказательные вероятности. Визуализация и диагностика сходимости (R-hat, трассировки) выполняются стандартными функциями PyMC.

**Предсказательное распределение (posterior predictive).** Для новой точки $\mathbf{x}_*$ вероятность $y_*=1$ оценивается как среднее по сэмплам:

$$
p(y_*=1 \mid \mathbf{x}_*, \mathbf{X}, \mathbf{y}) \approx \frac{1}{S} \sum_{s=1}^S \sigma\bigl(\mathbf{x}_*^T \mathbf{w}^{(s)}\bigr),
$$

где $\mathbf{w}^{(s)} \sim p(\mathbf{w} \mid \mathbf{X}, \mathbf{y})$ — сэмплы из апостериорного распределения.

#### 5. Мост к латентным переменным и EM-алгоритму

Иерархические модели и представление через масштабные смеси естественно подводят к общему классу моделей с латентными переменными. В контексте логистической регрессии уже упоминалась пробит-модель с латентной $z_i \sim \mathcal{N}(\mathbf{x}_i^T \mathbf{w}, 1)$ и $y_i = \mathbf{1}_{\{z_i > 0\}}$. Её байесовский вывод также может осуществляться через EM-алгоритм, где E-шаг вычисляет моменты усечённого нормального распределения, а M-шаг решает взвешенную линейную регрессию. Это связывает материал текущей главы с последующими темами: смеси распределений, скрытые марковские модели и более общие байесовские сети.

Байесовская логистическая регрессия, таким образом, не только является самостоятельным инструментом классификации с полным учётом неопределённости, но и служит фундаментом для понимания современных иерархических моделей и методов их вывода.

---

#### Вопросы для самопроверки (по всей главе «Байесовская логистическая регрессия»)

**Для начинающих**
1. Зачем использовать иерархические априорные распределения на $\sigma_w^2$? Что они дают по сравнению с фиксированным параметром?
2. Что такое робастное априорное распределение? Приведите пример распределения, делающего модель менее чувствительной к большим коэффициентам.
3. Как MCMC (в частности, NUTS) помогает в байесовском выводе для логистической регрессии?
4. В чём преимущество байесовской логистической регрессии перед обычной при наличии полной разделимости классов?
5. Что такое WAIC и как он используется для сравнения байесовских моделей?
6. Какую роль играют латентные переменные в байесовском моделировании? Приведите пример из пробит-регрессии.

**Для профессионалов**
1. Выведите выражение для posterior predictive распределения в байесовской логистической регрессии через MCMC-сэмплы. Как оценивается его неопределённость?
2. Объясните, как выбрать гиперприор для $\sigma_w^2$: сравните свойства $\operatorname{Inv-Gamma}(a,b)$ и $\operatorname{Half-Cauchy}(\beta)$. В чём преимущество полукоши?
3. Сравните эффективность лапласовского приближения и полного MCMC для логистической регрессии при $n=100, d=10$. В каких ситуациях разница между ними становится существенной?
4. Покажите формально, что любое собственное априорное распределение гарантирует конечность MAP-оценки и решает проблему разделимости. Как это соотносится с методом коррекции смещения Ферта?
5. Используя библиотеку NumPyro, реализуйте байесовскую модель лассо для логистической регрессии и сравните скорость сэмплирования на GPU с реализацией на PyMC.